# Pairloom — pretrained ESM-2 fine-tuning

The default experiment starts from **Meta's original ESM-2 35M weights** (`facebook/esm2_t12_35M_UR50D`), then learns protein interactions using Pairloom's own code and data. It does not use PLM-interact's interaction-trained weights or the failed RIMNet checkpoint. All encoder layers, embeddings, language-model readout and the new interaction head are trainable.

Set `INITIALIZATION = "random"` for the scratch-training control, using a separate run name and matching the other settings. Set `ARCHITECTURE = "esm2-650m"` to select Meta's larger 650M model. Keep 35M for the first Colab experiment. Source model identifiers, resolved revisions and initial weight hashes are recorded. A failed/incomplete pretrained download stops clearly; it never falls back to random weights.

Select a GPU runtime and run the cells in order. Supply a development CSV/CSV.gz or the existing `rimnet-kaggle-upload-v1.zip` in **My Drive/RIMNet/**. Only the development data and manifest are read from that ZIP; its weights, checkpoints and training wheels are ignored. Meta's pretrained weights download separately on the first run, unless `PRETRAINED_MODEL` names a local ESM-2 model directory.

Outputs go to a new folder under **My Drive/Pairloom/**. Later sessions resume their full Pairloom checkpoint without downloading the base weights again. Evaluation and inference also use only the full checkpoint. Versions before this full-model format are not silently reused. Changing scientific settings needs a new run name.

Pairloom uses the supplied data splits as provided, without checking protein overlap, repeated/reversed pairs or family separation. Prepare the desired splits before training; basic input and class-label checks still apply.

Full validation selects checkpoints every 1,000 updates and at epoch ends. Latest/previous/best include the complete model, optimizer and RNG state and are checksum-verified before Drive pointers are updated. Window bags have bounded coverage. The three-epoch default is an initial budget, not proof of adequate training. Software checks do not establish predictive quality or an improvement over RIMNet/PLM-interact.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

INPUT_DATA = Path("/content/drive/MyDrive/RIMNet/rimnet-kaggle-upload-v1.zip")
RUN_NAME = "pretrained-esm2-35m-cls-seed17"
INITIALIZATION = "pretrained"   # "random" is the scratch-training control.
PRETRAINED_MODEL = None        # Optional local Meta ESM-2 directory.
REVISION = None                # Optional pinned Hugging Face revision.
ARCHITECTURE = "esm2-35m"       # "esm2-650m" requires more training memory/compute.
PERSISTENT_RUN = Path("/content/drive/MyDrive/Pairloom") / RUN_NAME
HEAD = "cls"                     # "cls" or "residue"
MLM_WEIGHT = 0.1
MAX_TOKENS = 1024
MAX_VIEWS = 2
SEED = 17
EPOCHS = 3
SESSION_HOURS = 6.0
MAX_UPDATES = None              # Optional absolute update limit; no fake research score.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers==5.12.1", "scikit-learn>=1.3,<2"], check=True)
import gc, torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before starting real training.")
SOURCE_ROOT = Path("/content/pairloom-code")
(SOURCE_ROOT / "pairloom").mkdir(parents=True, exist_ok=True)
SOURCES = {'__init__.py': '"""Pairloom: protein-pair learning with pretrained and random ESM-2 controls."""\n__version__ = "0.3.0"\n', '__main__.py': 'from .cli import main\n\nif __name__ == "__main__":\n    main()\n', 'base.py': '"""Meta ESM-2 initialization, optional random controls and data-only imports."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport torch\n\n# ESM-2\'s published alphabet is a fixed token vocabulary, not learned weights.\nTOKENS = (\'<cls>\', \'<pad>\', \'<eos>\', \'<unk>\', \'L\', \'A\', \'G\', \'V\', \'S\', \'E\', \'R\', \'T\', \'I\',\n          \'D\', \'P\', \'K\', \'Q\', \'N\', \'F\', \'Y\', \'M\', \'H\', \'W\', \'C\', \'X\', \'B\', \'U\', \'Z\', \'O\',\n          \'.\', \'-\', \'<null_1>\', \'<mask>\')\nARCHITECTURES = {"esm2-35m": (12, 480, 20), "esm2-650m": (33, 1280, 20)}\nPRETRAINED_MODELS = {"esm2-35m": "facebook/esm2_t12_35M_UR50D",\n                     "esm2-650m": "facebook/esm2_t33_650M_UR50D"}\n\n\ndef weight_digest(model):\n    digest = hashlib.sha256()\n    for name, tensor in model.state_dict().items():\n        digest.update(name.encode() + tensor.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef initialize_architecture(architecture="esm2-35m", seed=17, config=None):\n    """Construct ESM-2 from a preset/local JSON, never a pretrained checkpoint.\n\n    Seeded construction runs in an isolated CPU RNG context. The recorded config\n    and initialization hash allow restoration without a model repository or base\n    weight file. All learned weights are subsequently included in checkpoints.\n    """\n    from transformers import EsmConfig, EsmForMaskedLM\n    if seed < 0:\n        raise ValueError("Initialization seed must be nonnegative.")\n    if config is None:\n        if architecture in ARCHITECTURES:\n            layers, width, heads = ARCHITECTURES[architecture]\n            config = {"num_hidden_layers": layers, "hidden_size": width,\n                      "num_attention_heads": heads, "intermediate_size": width * 4}\n        else:\n            path = Path(architecture)\n            if not path.is_file():\n                raise ValueError("Choose esm2-35m, esm2-650m or a local ESM-2 architecture JSON.")\n            config = json.loads(path.read_text())\n    values = {"vocab_size": len(TOKENS), "pad_token_id": 1, "mask_token_id": 32,\n              "position_embedding_type": "rotary", "token_dropout": True,\n              "hidden_dropout_prob": 0.0, "attention_probs_dropout_prob": 0.0,\n              "layer_norm_eps": 1e-5, "max_position_embeddings": 1026, **config}\n    if (values["vocab_size"] != len(TOKENS) or values["pad_token_id"] != 1\n            or values["mask_token_id"] != 32 or values["position_embedding_type"] != "rotary"):\n        raise ValueError("Expected the fixed ESM-2 alphabet and rotary architecture.")\n    if (values["num_hidden_layers"] < 1 or values["hidden_size"] < 1\n            or values["num_attention_heads"] < 1\n            or values["hidden_size"] % values["num_attention_heads"]):\n        raise ValueError("Invalid ESM-2 architecture dimensions.")\n    with torch.random.fork_rng(devices=[]):\n        torch.random.default_generator.manual_seed(seed)\n        model = EsmForMaskedLM(EsmConfig(**values))\n    identity = {"kind": "random-esm2", "config": values, "seed": seed,\n                "initial_weights_sha256": weight_digest(model)}\n    return model, {token: index for index, token in enumerate(TOKENS)}, identity\n\n\ndef initialize_pretrained(architecture="esm2-35m", source=None, revision=None, local_files_only=False):\n    """Load a complete Meta ESM-2 MLM checkpoint, never PLM-interact\'s weights.\n\n    A local ESM-2 directory is accepted for offline operation/testing. Loading\n    errors or missing learned tensors fail explicitly instead of falling back to\n    random initialization. Model files use safetensors.\n    """\n    from transformers import EsmForMaskedLM\n    if source is None:\n        if architecture not in PRETRAINED_MODELS:\n            raise ValueError("Pretrained initialization needs esm2-35m/esm2-650m or an explicit local model directory.")\n        source = PRETRAINED_MODELS[architecture]\n    if not Path(source).is_dir() and source not in PRETRAINED_MODELS.values():\n        raise ValueError("Choose an official Meta ESM-2 repository or a local ESM-2 directory.")\n    model, loading = EsmForMaskedLM.from_pretrained(\n        str(source), revision=revision, local_files_only=local_files_only,\n        use_safetensors=True, output_loading_info=True)\n    if any(loading.get(key) for key in ["missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs"]):\n        raise ValueError("The pretrained ESM-2 checkpoint did not load completely: " + str(loading))\n    config = model.config.to_dict()\n    if (config["vocab_size"] != len(TOKENS) or config["pad_token_id"] != 1\n            or config["mask_token_id"] != 32 or config["position_embedding_type"] != "rotary"):\n        raise ValueError("Expected an ESM-2 checkpoint with its fixed alphabet and rotary architecture.")\n    identity = {"kind": "pretrained-esm2", "source": str(source),\n                "revision": getattr(model.config, "_commit_hash", None) or revision,\n                "config": config, "initial_weights_sha256": weight_digest(model)}\n    return model, {token: index for index, token in enumerate(TOKENS)}, identity\n\n\ndef verified_path(root, item):\n    path = (Path(root) / item["path"]).resolve()\n    if not path.is_relative_to(Path(root).resolve()) or sha256(path) != item["sha256"]:\n        raise ValueError("Data path or checksum mismatch: " + item["path"])\n    return path\n\n\ndef extract_development_data(path, destination):\n    """Read only the manifest and development CSV from an old transfer archive.\n\n    Pretrained weights, old checkpoints and training wheels are neither read nor\n    extracted. A data-only archive with the same manifest structure also works.\n    """\n    path, destination = Path(path), Path(destination).resolve()\n    if path.is_dir():\n        manifest = json.loads((path / "transfer.json").read_text())\n        return verified_path(path, manifest["data"])\n    with zipfile.ZipFile(path) as archive:\n        manifests = [name for name in archive.namelist() if Path(name).name == "transfer.json"]\n        if len(manifests) != 1:\n            raise ValueError("Expected exactly one data manifest in the archive.")\n        manifest_name = manifests[0]\n        manifest = json.loads(archive.read(manifest_name))\n        relative = Path(manifest["data"]["path"])\n        if relative.is_absolute() or ".." in relative.parts or ".." in Path(manifest_name).parts:\n            raise ValueError("Unsafe data archive path.")\n        member_name = (Path(manifest_name).parent / relative).as_posix()\n        member = archive.getinfo(member_name)\n        if (member.external_attr >> 16) & 0o170000 == 0o120000:\n            raise ValueError("The development data must not be a symbolic link.")\n        # Isolate datasets by their declared content hash, not by model files.\n        checksum = manifest["data"]["sha256"]\n        if len(checksum) != 64 or any(c not in "0123456789abcdef" for c in checksum):\n            raise ValueError("Expected a SHA-256 data checksum.")\n        root = destination / checksum\n        target = (root / relative).resolve()\n        if not target.is_relative_to(root.resolve()):\n            raise ValueError("Unsafe data destination.")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        if not target.exists() or sha256(target) != manifest["data"]["sha256"]:\n            temporary = target.with_suffix(target.suffix + ".partial")\n            with archive.open(member) as source, temporary.open("wb") as output:\n                for block in iter(lambda: source.read(8 * 1024**2), b""):\n                    output.write(block)\n            if sha256(temporary) != manifest["data"]["sha256"]:\n                raise ValueError("Development data checksum mismatch.")\n            temporary.replace(target)\n    return target\n', 'checkpoints.py': '"""Versioned checkpoint files with atomic latest/best pointers and verified backups."""\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\n\nimport numpy as np\nimport torch\nfrom .base import sha256\n\nFORMAT = "pairloom-full-model-v1"\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w") as stream:\n        json.dump(value, stream, indent=2, allow_nan=False)\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef rng_state():\n    state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}\n    if torch.cuda.is_available():\n        state["cuda"] = torch.cuda.get_rng_state_all()\n    if torch.backends.mps.is_available():\n        state["mps"] = torch.mps.get_rng_state()\n    return state\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch"])\n    if "cuda" in state and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state["cuda"])\n    if "mps" in state and torch.backends.mps.is_available():\n        torch.mps.set_rng_state(state["mps"])\n\n\nclass CheckpointStore:\n    def __init__(self, root):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n\n    def save(self, value, best=False):\n        import uuid\n        filename = f"step-{value[\'step\']:08d}-{uuid.uuid4().hex[:8]}.pt"\n        path = self.root / filename\n        temporary = path.with_suffix(".tmp")\n        with temporary.open("wb") as stream:\n            torch.save({**value, "format": FORMAT}, stream)\n            stream.flush()\n            os.fsync(stream.fileno())\n        temporary.replace(path)\n        pointer = {"file": filename, "sha256": sha256(path), "step": value["step"]}\n        latest = self.root / "latest.json"\n        if latest.is_file():\n            old = json.loads(latest.read_text())\n            old_path = self.root / old["file"]\n            if old_path.exists() and sha256(old_path) == old["sha256"]:\n                atomic_json(self.root / "previous.json", old)\n        atomic_json(latest, pointer)\n        if best:\n            atomic_json(self.root / "best.json", pointer)\n        # Only this store\'s unreferenced, generated step files may be pruned.\n        keep = {json.loads(p.read_text())["file"] for p in self.root.glob("*.json")\n                if p.name in {"latest.json", "previous.json", "best.json"}}\n        for old in self.root.glob("step-*.pt"):\n            if old.name not in keep:\n                old.unlink()\n        return path\n\n    def load(self, which="latest", fallback=True):\n        failures = []\n        for name in ([which, "previous"] if which == "latest" and fallback else [which]):\n            try:\n                pointer = json.loads((self.root / (name + ".json")).read_text())\n                path = (self.root / pointer["file"]).resolve()\n                if not path.is_relative_to(self.root.resolve()) or sha256(path) != pointer["sha256"]:\n                    raise ValueError("Checkpoint checksum or path mismatch.")\n                saved = torch.load(path, map_location="cpu", weights_only=False)\n                if saved.get("format") != FORMAT or saved["step"] != pointer["step"]:\n                    raise ValueError("Checkpoint identity mismatch.")\n                if name != which:\n                    print("Recovered previous verified checkpoint at step", saved["step"], flush=True)\n                return saved\n            except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:\n                failures.append(f"{name}: {exc}")\n        raise RuntimeError("No valid checkpoint. " + "; ".join(failures))\n\n    def mirror(self, destination):\n        """Copy selected checkpoints to persistent storage before publishing pointers."""\n        destination = Path(destination)\n        destination.mkdir(parents=True, exist_ok=True)\n        if destination.resolve() == self.root.resolve():\n            return\n        source_run, target_run = self.root / "run.json", destination / "run.json"\n        if source_run.is_file() and target_run.is_file():\n            if json.loads(source_run.read_text())["protocol"] != json.loads(target_run.read_text())["protocol"]:\n                raise ValueError("Persistent destination belongs to a different experiment.")\n        for name in ["latest", "previous", "best"]:\n            pointer_file = self.root / (name + ".json")\n            if not pointer_file.exists():\n                continue\n            pointer = json.loads(pointer_file.read_text())\n            source, target = self.root / pointer["file"], destination / pointer["file"]\n            if sha256(source) != pointer["sha256"]:\n                raise ValueError("Refusing to mirror an invalid checkpoint.")\n            if not target.exists() or sha256(target) != pointer["sha256"]:\n                temporary = target.with_suffix(".partial")\n                shutil.copyfile(source, temporary)\n                if sha256(temporary) != pointer["sha256"]:\n                    raise OSError("Persistent checkpoint copy failed verification.")\n                temporary.replace(target)\n            atomic_json(destination / pointer_file.name, pointer)\n        for name in ["run.json", "history.json", "validation.csv"]:\n            source = self.root / name\n            if source.exists():\n                temporary = destination / (name + ".partial")\n                shutil.copyfile(source, temporary)\n                temporary.replace(destination / name)\n        keep = {json.loads(p.read_text())["file"] for p in destination.glob("*.json")\n                if p.name in {"latest.json", "previous.json", "best.json"}}\n        for old in destination.glob("step-*.pt"):\n            if old.name not in keep:\n                old.unlink()\n', 'cli.py': '"""Pairloom\'s train, evaluate and predict command line."""\nimport argparse\nfrom dataclasses import asdict, fields\nimport fcntl\nimport json\nfrom pathlib import Path\n\nfrom . import __version__\n\n\ndef parser():\n    result = argparse.ArgumentParser(prog="pairloom", description="Train protein-pair representations with ESM-2.")\n    result.add_argument("--version", action="version", version=__version__)\n    commands = result.add_subparsers(dest="command", required=True)\n    for name in ["train", "evaluate", "predict"]:\n        command = commands.add_parser(name)\n        command.add_argument("--data", required=True, help="CSV or CSV.gz; RIMNet or query/text columns.")\n        command.add_argument("--task", choices=["ppi", "mutation"], default="ppi")\n        command.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda", "mps"])\n        command.add_argument("--precision", default="auto", choices=["auto", "fp32", "fp16", "bf16"])\n        if name == "train":\n            command.add_argument("--initialization", choices=["pretrained", "random"], help="Default: pretrained Meta ESM-2; random is a scratch-training control.")\n            command.add_argument("--architecture", help="esm2-35m (default), esm2-650m; random mode also accepts a local architecture JSON.")\n            command.add_argument("--pretrained-model", help="Optional local ESM-2 directory or official Meta repository override.")\n            command.add_argument("--revision", help="Optional Hugging Face revision; resolved provenance is recorded.")\n            command.add_argument("--output", required=True)\n            command.add_argument("--validation-data", help="Separate validation CSV; otherwise use data\'s split column.")\n            command.add_argument("--resume", action="store_true")\n            command.add_argument("--persistent", help="Mirror verified latest/previous/best checkpoints, e.g. Drive.")\n            command.add_argument("--max-updates", type=int, help="Pause at this absolute optimizer update.")\n            command.add_argument("--hours", type=float, help="Wall-clock budget; does not alter the scientific protocol.")\n            from .engine import TrainConfig\n            defaults = TrainConfig()\n            for field in fields(defaults):\n                value = getattr(defaults, field.name)\n                option = "--" + field.name.replace("_", "-")\n                if isinstance(value, bool):\n                    command.add_argument(option, action=argparse.BooleanOptionalAction, default=None)\n                else:\n                    command.add_argument(option, type=type(value) if value is not None else float, default=None,\n                                         help=f"Default: {value}")\n        else:\n            command.add_argument("--run", required=True, help="Pairloom checkpoint directory.")\n            command.add_argument("--checkpoint", default="best", choices=["best", "latest", "previous"])\n            command.add_argument("--split", help="Select a named split; no filtering if omitted.")\n            command.add_argument("--predictions", required=True)\n    return result\n\n\ndef execution_device(name, precision):\n    import torch\n    if name == "auto":\n        name = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"\n    device = torch.device(name)\n    if name == "cuda" and not torch.cuda.is_available():\n        raise ValueError("CUDA is unavailable.")\n    if name == "mps" and not torch.backends.mps.is_available():\n        raise ValueError("MPS is unavailable.")\n    if precision == "auto":\n        precision = "fp32"\n        if name == "cuda":\n            precision = "bf16" if torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported() else "fp16"\n    return device, precision\n\n\ndef make_model(args, config, expected=None):\n    from .base import initialize_architecture, initialize_pretrained\n    from .engine import seed_everything\n    from .model import PairModel\n    from .data import PairBatcher\n    selected = getattr(args, "architecture", None)\n    mode = getattr(args, "initialization", None)\n    source, revision = getattr(args, "pretrained_model", None), getattr(args, "revision", None)\n    if expected is not None:\n        if expected.get("kind") not in {"random-esm2", "pretrained-esm2"}:\n            raise ValueError("Unsupported Pairloom initialization provenance.")\n        if any(value is not None for value in [selected, mode, source, revision]):\n            raise ValueError("When resuming, omit initialization/model flags; the full checkpoint defines them.")\n        # Full checkpoints restore every model tensor. No base-model download is\n        # needed for resume/evaluation, even if the original cache was removed.\n        backbone, vocab, _ = initialize_architecture(seed=config.seed, config=expected["config"])\n        identity = expected\n    elif (mode or "pretrained") == "pretrained":\n        backbone, vocab, identity = initialize_pretrained(selected or "esm2-35m", source, revision)\n    else:\n        if source is not None or revision is not None:\n            raise ValueError("Random initialization does not accept a pretrained model or revision.")\n        backbone, vocab, identity = initialize_architecture(selected or "esm2-35m", config.seed)\n    seed_everything(config.seed)\n    model = PairModel(backbone, config.head, config.head_width, config.dropout, config.gradient_checkpointing)\n    batcher = PairBatcher(vocab, config.max_tokens, config.max_views, config.seed)\n    return model, batcher, identity\n\n\ndef run(args):\n    from .checkpoints import CheckpointStore\n    from .data import read_records\n    from .engine import TrainConfig, fit, predict, write_predictions, source_identity\n    from .metrics import report\n    device, precision = execution_device(args.device, args.precision)\n    if args.command == "train":\n        output = Path(args.output)\n        output.mkdir(parents=True, exist_ok=True)\n        # The lock spans model loading, training and checkpoint publication.\n        with (output / "run.lock").open("a") as lock:\n            try:\n                fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n            except BlockingIOError:\n                raise ValueError("Another process is using this run directory.")\n            saved = CheckpointStore(output).load() if args.resume else None\n            options = saved["protocol"]["config"].copy() if saved else asdict(TrainConfig())\n            for key in options:\n                if getattr(args, key) is not None:\n                    options[key] = getattr(args, key)\n            config = TrainConfig(**options)\n            config.validate()\n            if args.max_updates is not None and args.max_updates < 1 or args.hours is not None and args.hours <= 0:\n                raise ValueError("Training budgets must be positive.")\n            training = read_records(args.data, None if args.validation_data else "train", args.task)\n            validation = read_records(args.validation_data or args.data, None if args.validation_data else "validation", args.task)\n            from .data import validate_training_labels\n            validate_training_labels(training, validation)\n            expected = saved["protocol"]["initialization"] if saved else None\n            del saved\n            model, batcher, identity = make_model(args, config, expected)\n            model.to(device)\n            return fit(model, batcher, training, validation, identity, config, output, device,\n                       precision, args.resume, args.max_updates, args.hours, args.persistent)\n    saved = CheckpointStore(args.run).load(args.checkpoint, fallback=False)\n    if saved["protocol"]["task"] != args.task:\n        raise ValueError("The requested task differs from this checkpoint\'s training task.")\n    if saved["protocol"]["source"] != source_identity():\n        raise ValueError("This checkpoint uses a different Pairloom implementation.")\n    config = TrainConfig(**saved["protocol"]["config"])\n    rows = read_records(args.data, args.split, args.task, labelled=args.command == "evaluate")\n    model, batcher, _ = make_model(args, config, saved["protocol"]["initialization"])\n    model.restore(saved["model"])\n    model.to(device)\n    scores = predict(model, batcher, rows, device, config.batch_size, precision, progress=True)\n    write_predictions(args.predictions, rows, scores)\n    result = {"checkpoint_step": saved["step"], "predictions": args.predictions, "records": len(rows)}\n    if args.command == "evaluate":\n        threshold = saved["state"]["threshold"]\n        if threshold is None:\n            raise ValueError("This checkpoint has no validation-selected threshold; evaluate a validated checkpoint.")\n        result["metrics"] = report([r.label for r in rows], scores, threshold)\n    return result\n\n\ndef main(argv=None):\n    arguments = parser().parse_args(argv)\n    try:\n        print(json.dumps(run(arguments), indent=2, allow_nan=False))\n    except (ValueError, OSError, RuntimeError) as exc:\n        raise SystemExit(str(exc)) from exc\n\n\nif __name__ == "__main__":\n    main()\n', 'data.py': '"""Sequence-pair records, supplied split selection and deterministic window bags."""\nfrom dataclasses import dataclass\nimport csv\nimport gzip\nimport hashlib\nimport random\n\nimport torch\n\nAMINO_ACIDS = frozenset("ACDEFGHIKLMNPQRSTVWYBXZUO")\n\n\n@dataclass(frozen=True)\nclass Record:\n    pair_id: str\n    sequence_a: str\n    sequence_b: str\n    label: int | None = None\n    mutant: str | None = None\n\n\ndef sequence(value):\n    value = "".join(value.upper().split())\n    if not value or not set(value) <= AMINO_ACIDS:\n        raise ValueError("Expected a nonempty single-chain amino-acid sequence.")\n    return value\n\n\ndef read_records(path, split=None, task="ppi", labelled=True):\n    """Read RIMNet columns or query/text aliases without changing split membership."""\n    opener = gzip.open if str(path).endswith(".gz") else open\n    records, identifiers = [], set()\n    with opener(path, "rt", newline="") as stream:\n        for row_number, row in enumerate(csv.DictReader(stream), start=1):\n            if split is not None and row.get("split") != split:\n                continue\n            if task == "mutation":\n                a, b = sequence(row["wild_seq"]), sequence(row["participant_sequence"])\n                mutant = sequence(row["mutant_seq"])\n            else:\n                a = sequence(row.get("sequence_a", row.get("query", "")))\n                b = sequence(row.get("sequence_b", row.get("text", "")))\n                mutant = None\n            raw = row.get("label", "")\n            labels = {"0": 0, "1": 1, "decreasing": 0, "increasing": 1}\n            if labelled and raw not in labels:\n                raise ValueError("Labels must be 0/1, or decreasing/increasing for mutations.")\n            label = labels.get(raw)\n            content = "|".join(sorted((a, b))) if mutant is None else "|".join((a, mutant, b))\n            # Distinct source rows remain distinct even when pairs repeat or\n            # appear in reversed order. Number before filtering for stable IDs.\n            pair_id = row.get("pair_id") or f"{hashlib.sha256(content.encode()).hexdigest()}:{row_number}"\n            if pair_id in identifiers:\n                raise ValueError("Duplicate pair_id: " + pair_id)\n            identifiers.add(pair_id)\n            records.append(Record(pair_id, a, b, label, mutant))\n    if not records:\n        raise ValueError("No records matched the requested input/split.")\n    return records\n\n\ndef fingerprint(records):\n    digest = hashlib.sha256()\n    for r in records:\n        digest.update(repr((r.pair_id, r.sequence_a, r.sequence_b, r.label, r.mutant)).encode() + b"\\n")\n    return digest.hexdigest()\n\n\ndef validate_training_labels(training, validation):\n    """Require classes needed for weighting and metrics; trust supplied splits."""\n    if {r.label for r in training} != {0, 1} or {r.label for r in validation} != {0, 1}:\n        raise ValueError("Training and validation must each contain both classes.")\n\n\ndef starts(length, width):\n    """Non-overlapping windows plus an end-aligned window when needed."""\n    if length <= width:\n        return [0]\n    return sorted(set([*range(0, length - width + 1, width), length - width]))\n\n\ndef window_coordinates(a, b, max_tokens, max_views, training=False, seed=0):\n    """Select bounded pairs of windows; only the whole bag receives the pair label."""\n    budget = max_tokens - 3\n    if budget < 2 or max_views < 1:\n        raise ValueError("max_tokens must be >=5 and max_views >=1.")\n    swapped = a > b\n    if swapped:\n        a, b = b, a\n    wa = min(len(a), max(1, budget // 2))\n    wb = min(len(b), budget - wa)\n    wa = min(len(a), budget - wb)\n    aa, bb = starts(len(a), wa), starts(len(b), wb)\n    total = len(aa) * len(bb)\n    count = min(max_views, total)\n    if training:\n        selected = sorted(random.Random(seed).sample(range(total), count))\n    elif count == 1:\n        selected = [total // 2]\n    else:\n        selected = [i * (total - 1) // (count - 1) for i in range(count)]\n    coordinates = [(aa[i // len(bb)], wa, bb[i % len(bb)], wb) for i in selected]\n    return [(b0, bw, a0, aw) for a0, aw, b0, bw in coordinates] if swapped else coordinates\n\n\ndef windows(a, b, max_tokens, max_views, training=False, seed=0):\n    return [(a[a0:a0 + aw], b[b0:b0 + bw])\n            for a0, aw, b0, bw in window_coordinates(a, b, max_tokens, max_views, training, seed)]\n\n\nclass PairBatcher:\n    def __init__(self, vocab, max_tokens=1024, max_views=2, seed=17):\n        self.vocab, self.max_tokens, self.max_views, self.seed = vocab, max_tokens, max_views, seed\n        self.residue_ids = torch.tensor([vocab[c] for c in sorted(AMINO_ACIDS) if c in vocab])\n\n    def __call__(self, records, training=False, epoch=0, reverse=False):\n        packed, chains, owners = [], [], []\n        mutation = records[0].mutant is not None\n        if any((r.mutant is not None) != mutation for r in records):\n            raise ValueError("PPI and mutation records cannot share a batch.")\n        for index, r in enumerate(records):\n            seed = int.from_bytes(hashlib.sha256(f"{self.seed}:{epoch}:{r.pair_id}".encode()).digest()[:8], "big")\n            wild_coordinates = window_coordinates(r.sequence_a, r.sequence_b, self.max_tokens, self.max_views, training, seed)\n            # Same-length variants retain exactly the wild-type crop coordinates.\n            for variant, a in enumerate([r.sequence_a, r.mutant] if mutation else [r.sequence_a]):\n                swapped = reverse if not training else bool(random.Random(seed).getrandbits(1))\n                coordinates = (wild_coordinates if len(a) == len(r.sequence_a) else\n                               window_coordinates(a, r.sequence_b, self.max_tokens, self.max_views, training, seed))\n                for a0, aw, b0, bw in coordinates:\n                    x, y = a[a0:a0 + aw], r.sequence_b[b0:b0 + bw]\n                    if swapped:\n                        x, y = y, x\n                    xi, yi = [self.vocab[c] for c in x], [self.vocab[c] for c in y]\n                    packed.append([self.vocab["<cls>"], *xi, self.vocab["<eos>"], *yi, self.vocab["<eos>"]])\n                    chains.append([0, *([1] * len(xi)), 0, *([2] * len(yi)), 0])\n                    owners.append(index * (2 if mutation else 1) + variant)\n        length = max(map(len, packed))\n        ids = torch.full((len(packed), length), self.vocab["<pad>"], dtype=torch.long)\n        chain = torch.zeros_like(ids)\n        for i, (tokens, groups) in enumerate(zip(packed, chains)):\n            ids[i, :len(tokens)] = torch.tensor(tokens)\n            chain[i, :len(groups)] = torch.tensor(groups)\n        return {"input_ids": ids, "attention_mask": ids.ne(self.vocab["<pad>"]), "chains": chain,\n                "owners": torch.tensor(owners), "groups": len(records) * (2 if mutation else 1),\n                "mutation": mutation, "labels": torch.tensor([r.label or 0 for r in records], dtype=torch.float32)}\n\n    def mask(self, batch, probability):\n        if not 0 < probability <= 1:\n            raise ValueError("Mask probability must be in (0,1].")\n        batch = dict(batch)\n        ids = batch["input_ids"].clone()\n        eligible = batch["chains"].ne(0)\n        chosen = (torch.rand(ids.shape) < probability) & eligible\n        if not chosen.any():\n            positions = eligible.nonzero()\n            pos = positions[torch.randint(len(positions), ())]\n            chosen[tuple(pos)] = True\n        targets = ids.clone().masked_fill(~chosen, -100)\n        draw = torch.rand(ids.shape)\n        ids[chosen & (draw < .8)] = self.vocab["<mask>"]\n        random_positions = chosen & (draw >= .8) & (draw < .9)\n        ids[random_positions] = self.residue_ids[torch.randint(len(self.residue_ids), (int(random_positions.sum()),))]\n        batch.update(input_ids=ids, mlm_targets=targets)\n        return batch\n', 'engine.py': '"""Single-device pair training with exact update boundaries and validation selection."""\nfrom contextlib import nullcontext\nfrom dataclasses import asdict, dataclass\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nfrom .checkpoints import CheckpointStore, atomic_json, restore_rng, rng_state\nfrom .data import fingerprint, validate_training_labels\nfrom .metrics import report, threshold_mcc\n\n\n@dataclass(frozen=True)\nclass TrainConfig:\n    epochs: int = 3\n    seed: int = 17\n    batch_size: int = 1\n    accumulation: int = 16\n    encoder_lr: float = 2e-5\n    head_lr: float = 1e-4\n    weight_decay: float = .01\n    warmup_updates: int = 100\n    mlm_weight: float = .1\n    mask_probability: float = .15\n    max_tokens: int = 1024\n    max_views: int = 2\n    head: str = "cls"\n    head_width: int = 128\n    dropout: float = .1\n    gradient_checkpointing: bool = True\n    eval_every: int = 1000\n    checkpoint_every: int = 25\n    positive_weight: float | None = None\n\n    def validate(self):\n        if any(not math.isfinite(value) for value in [self.encoder_lr, self.head_lr, self.weight_decay,\n                self.mlm_weight, self.mask_probability, self.dropout]):\n            raise ValueError("Numeric training settings must be finite.")\n        if min(self.epochs, self.batch_size, self.accumulation, self.max_views, self.eval_every, self.checkpoint_every) < 1:\n            raise ValueError("Epoch, batch, view and interval settings must be positive.")\n        if self.max_tokens < 5 or self.head_width % 4 or self.head_width < 4:\n            raise ValueError("Invalid context length or head width.")\n        if not 0 <= self.dropout < 1 or not 0 < self.mask_probability <= 1 or self.mlm_weight < 0:\n            raise ValueError("Invalid dropout or masking/loss settings.")\n        if min(self.encoder_lr, self.head_lr) <= 0 or self.weight_decay < 0 or self.warmup_updates < 0:\n            raise ValueError("Invalid optimizer settings.")\n        if self.positive_weight is not None and (not math.isfinite(self.positive_weight) or self.positive_weight <= 0):\n            raise ValueError("positive_weight must be positive.")\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef source_identity():\n    digest = hashlib.sha256()\n    for path in sorted(Path(__file__).parent.glob("*.py")):\n        digest.update(path.name.encode() + path.read_bytes())\n    return digest.hexdigest()\n\n\ndef mixed_precision(device, precision):\n    if precision == "fp32":\n        return nullcontext()\n    if device.type != "cuda" or precision not in {"fp16", "bf16"}:\n        raise ValueError("Use fp32 on CPU/MPS and fp16 or bf16 on CUDA.")\n    if precision == "bf16" and not torch.cuda.is_bf16_supported():\n        raise ValueError("This GPU does not support BF16.")\n    return torch.autocast("cuda", dtype=torch.float16 if precision == "fp16" else torch.bfloat16)\n\n\ndef to_device(batch, device):\n    return {key: value.to(device) if isinstance(value, torch.Tensor) else value for key, value in batch.items()}\n\n\ndef predict(model, batcher, records, device, batch_size=1, precision="fp32", deadline=None, progress=False):\n    model.eval()\n    scores = []\n    started = time.monotonic()\n    with torch.inference_mode():\n        for start in range(0, len(records), batch_size):\n            if deadline is not None and time.monotonic() >= deadline:\n                return None\n            rows = records[start:start + batch_size]\n            with mixed_precision(device, precision):\n                forward, _ = model(to_device(batcher(rows), device))\n                reverse, _ = model(to_device(batcher(rows, reverse=True), device))\n            probabilities = torch.sigmoid((forward.float() + reverse.float()) * .5)\n            scores.extend(probabilities.cpu().tolist())\n            if progress and (len(scores) % 128 < batch_size or len(scores) == len(records)):\n                elapsed = time.monotonic() - started\n                print(json.dumps({"event": "validation", "pairs": len(scores), "total": len(records),\n                                  "remaining_seconds": elapsed / len(scores) * (len(records) - len(scores))}), flush=True)\n    return np.asarray(scores)\n\n\ndef write_predictions(path, records, scores):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["pair_id", "label", "score"])\n        writer.writerows((r.pair_id, "" if r.label is None else r.label, float(s)) for r, s in zip(records, scores))\n    temporary.replace(path)\n\n\ndef fit(model, batcher, training, validation, initialization, config, output, device,\n        precision="fp32", resume=False, max_updates=None, hours=None, persistent=None):\n    """Fit all records; max_updates/hours pause the unchanged scientific protocol."""\n    config.validate()\n    validate_training_labels(training, validation)\n    if initialization.get("kind") not in {"random-esm2", "pretrained-esm2"}:\n        raise ValueError("Expected explicit random or pretrained ESM-2 provenance.")\n    if any(getattr(config, key) != value for key, value in model.settings.items()):\n        raise ValueError("The model architecture/trainability differs from TrainConfig.")\n    output = Path(output)\n    store = CheckpointStore(output)\n    if not resume and (output / "latest.json").exists():\n        raise FileExistsError("This run already has a checkpoint; resume it or use a new directory.")\n    if not resume and any(path.name != "run.lock" for path in output.iterdir()):\n        raise FileExistsError("A new run requires an empty output directory; preserve existing files and choose a new run name.")\n    protocol = {"config": asdict(config), "training": fingerprint(training), "validation": fingerprint(validation),\n                "initialization": initialization, "source": source_identity(),\n                "task": "mutation" if training[0].mutant is not None else "ppi",\n                "libraries": {name: importlib.metadata.version(name) for name in ["torch", "transformers"]}}\n    if (batcher.max_tokens, batcher.max_views, batcher.seed) != (config.max_tokens, config.max_views, config.seed):\n        raise ValueError("Batcher settings differ from the run configuration.")\n    trained = {name: p for name, p in model.named_parameters() if p.requires_grad}\n    groups = [{"params": [p for name, p in trained.items() if name.startswith("backbone.") == encoder],\n               "lr": lr, "initial_lr": lr} for encoder, lr in [(True, config.encoder_lr), (False, config.head_lr)]]\n    optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=config.weight_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and precision == "fp16")\n    seed_everything(config.seed)\n    state = {"epoch": 0, "cursor": 0, "step": 0, "history": [], "best_auroc": -1.,\n             "threshold": None, "pending_validation": False, "last_validation_step": -1}\n    if resume:\n        saved = store.load()\n        if saved["protocol"] != protocol:\n            raise ValueError("Resume initialization, code, data, libraries or scientific configuration changed.")\n        model.restore(saved["model"])\n        optimizer.load_state_dict(saved["optimizer"])\n        if scaler.is_enabled() and saved["scaler"]:\n            scaler.load_state_dict(saved["scaler"])\n        state = saved["state"]\n        restore_rng(saved["rng"])\n        del saved\n    positive_weight = config.positive_weight\n    if positive_weight is None:\n        positives = sum(r.label for r in training)\n        positive_weight = (len(training) - positives) / positives\n    weight = torch.tensor(positive_weight, device=device)\n    effective = config.batch_size * config.accumulation\n    total_updates = math.ceil(len(training) / effective) * config.epochs\n    start_time = time.monotonic()\n    deadline = start_time + hours * 3600 if hours else None\n    atomic_json(output / "run.json", {"protocol": protocol, "positive_weight": positive_weight,\n                "training_pairs": len(training), "validation_pairs": len(validation),\n                "device": str(device), "precision": precision, "total_updates": total_updates,\n                "evidence": "Development validation; no final-test selection or guaranteed performance.",\n                "window_limit": "max_views bounds coverage; long pairs may have unobserved window combinations."})\n\n    def checkpoint(best=False):\n        atomic_json(output / "history.json", state["history"])\n        store.save({"step": state["step"], "protocol": protocol, "model": model.checkpoint_state(),\n                    "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),\n                    "state": dict(state), "rng": rng_state()}, best)\n        if persistent:\n            store.mirror(persistent)\n\n    def validate():\n        scores = predict(model, batcher, validation, device, config.batch_size, precision, deadline, True)\n        if scores is None:\n            return False\n        threshold = threshold_mcc([r.label for r in validation], scores)\n        metrics = report([r.label for r in validation], scores, threshold)\n        state["history"].append({"step": state["step"], "epoch": state["epoch"], "validation": metrics})\n        state["threshold"] = threshold\n        state["pending_validation"] = False\n        state["last_validation_step"] = state["step"]\n        best = metrics["auroc"] > state["best_auroc"]\n        if best:\n            state["best_auroc"] = metrics["auroc"]\n            write_predictions(output / "validation.csv", validation, scores)\n        checkpoint(best)\n        print(json.dumps({"event": "validation_complete", "step": state["step"], "best": best, **metrics}), flush=True)\n        return True\n\n    status = "complete"\n    cached_epoch, order = None, None\n    # An initial checkpoint makes an interrupted first session recoverable.\n    if not resume:\n        checkpoint()\n    try:\n        while state["epoch"] < config.epochs:\n            if state["pending_validation"]:\n                if not validate():\n                    status = "paused_validation"\n                    break\n                continue\n            if state["cursor"] == len(training):\n                state["epoch"] += 1\n                state["cursor"] = 0\n                continue\n            if (max_updates is not None and state["step"] >= max_updates) or (deadline and time.monotonic() >= deadline):\n                status = "paused"\n                break\n            if cached_epoch != state["epoch"]:\n                order = np.random.default_rng(config.seed + state["epoch"]).permutation(len(training))\n                cached_epoch = state["epoch"]\n            selected = order[state["cursor"]:state["cursor"] + effective]\n            batches = [batcher([training[i] for i in selected[start:start + config.batch_size]],\n                               training=True, epoch=state["epoch"]) for start in range(0, len(selected), config.batch_size)]\n            if config.mlm_weight:\n                batches = [batcher.mask(batch, config.mask_probability) for batch in batches]\n            masked = sum(int(batch["mlm_targets"].ne(-100).sum()) for batch in batches) if config.mlm_weight else 1\n            factor = min((state["step"] + 1) / max(1, config.warmup_updates), 1.)\n            if state["step"] >= config.warmup_updates:\n                factor = max(0., (total_updates - state["step"]) / max(1, total_updates - config.warmup_updates))\n            for group in optimizer.param_groups:\n                group["lr"] = group["initial_lr"] * factor\n            attempt_rng = rng_state()\n            for attempt in range(8):\n                model.train()\n                optimizer.zero_grad(set_to_none=True)\n                class_total = mlm_total = 0.\n                for batch in batches:\n                    batch = to_device(batch, device)\n                    with mixed_precision(device, precision):\n                        logits, mlm_sum = model(batch)\n                        classification = F.binary_cross_entropy_with_logits(logits.float(), batch["labels"], pos_weight=weight, reduction="sum")\n                        loss = classification / len(selected) + config.mlm_weight * mlm_sum / masked\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError("Nonfinite training loss; last committed checkpoint is preserved.")\n                    scaler.scale(loss).backward()\n                    class_total += float(classification.detach())\n                    mlm_total += float(mlm_sum.detach())\n                scaler.unscale_(optimizer)\n                bad_gradient = any(p.grad is not None and not torch.isfinite(p.grad).all() for p in trained.values())\n                if bad_gradient and scaler.is_enabled():\n                    # GradScaler skips this update and lowers its scale. Replay\n                    # the same examples, masks and dropout; consume no progress.\n                    scaler.step(optimizer)\n                    scaler.update()\n                    restore_rng(attempt_rng)\n                    print(json.dumps({"event": "amp_retry", "step": state["step"],\n                                      "attempt": attempt + 1, "scale": scaler.get_scale()}), flush=True)\n                    continue\n                norm = torch.nn.utils.clip_grad_norm_(trained.values(), 1.)\n                if bad_gradient or not torch.isfinite(norm):\n                    raise FloatingPointError("Nonfinite gradients; last committed checkpoint is preserved.")\n                scaler.step(optimizer)\n                scaler.update()\n                break\n            else:\n                raise FloatingPointError("AMP overflow persisted after eight retries; use BF16/FP32 or review the learning rate.")\n            state["cursor"] += len(selected)\n            state["step"] += 1\n            state["pending_validation"] = state["cursor"] == len(training) or state["step"] % config.eval_every == 0\n            print(json.dumps({"event": "training", "step": state["step"], "total_updates": total_updates,\n                              "epoch": state["epoch"] + 1, "pairs_in_update": len(selected),\n                              "classification_loss": class_total / len(selected), "mlm_loss": mlm_total / masked,\n                              "elapsed_seconds": time.monotonic() - start_time}), flush=True)\n            if state["pending_validation"] or state["step"] % config.checkpoint_every == 0:\n                checkpoint()\n        checkpoint()\n    except KeyboardInterrupt:\n        # A partial optimizer update cannot safely be committed. Keep the last\n        # on-disk update, whose model/optimizer/RNG position is internally consistent.\n        status = "interrupted"\n        if persistent:\n            store.mirror(persistent)\n        print("Interrupted; use latest.json to resume the last committed update.", flush=True)\n    return {"status": status, "step": store.load()["step"], "best_validation_auroc": state["best_auroc"] if state["best_auroc"] >= 0 else None,\n            "output": str(output), "elapsed_seconds": time.monotonic() - start_time}\n', 'metrics.py': '"""Validation threshold selection and fixed-threshold reporting."""\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score\n\n\ndef threshold_mcc(labels, scores):\n    labels, scores = np.asarray(labels), np.asarray(scores)\n    if not len(labels) or not np.isfinite(scores).all() or set(labels) != {0, 1}:\n        raise ValueError("Threshold selection needs finite scores and both validation classes.")\n    order = np.argsort(scores, kind="stable")\n    positives = np.r_[0, np.cumsum(labels[order])]\n    candidates = np.unique(np.r_[0., scores, np.nextafter(scores.max(), np.inf)])\n    below = np.searchsorted(scores[order], candidates, side="left")\n    fn, tn = positives[below], below - positives[below]\n    tp, fp = positives[-1] - fn, len(labels) - below - positives[-1] + fn\n    denominator = np.sqrt((tp + fp).astype(float) * (tp + fn) * (tn + fp) * (tn + fn))\n    mcc = np.divide(tp * tn - fp * fn, denominator, out=np.zeros_like(denominator), where=denominator > 0)\n    tied = np.flatnonzero(mcc == mcc.max())\n    return float(candidates[tied[np.argmin(np.abs(candidates[tied] - .5))]])\n\n\ndef report(labels, scores, threshold):\n    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)\n    if len(labels) != len(scores) or not len(labels) or not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):\n        raise ValueError("Expected one finite probability per labelled record.")\n    predicted, positive = scores >= threshold, labels == 1\n    tp, tn = int((predicted & positive).sum()), int((~predicted & ~positive).sum())\n    fp, fn = int((predicted & ~positive).sum()), int((~predicted & positive).sum())\n    both = len(set(labels)) == 2\n    precision = tp / (tp + fp) if tp + fp else 0.\n    recall = tp / (tp + fn) if tp + fn else None\n    specificity = tn / (tn + fp) if tn + fp else None\n    clipped = np.clip(scores, 1e-7, 1 - 1e-7)\n    return {"n": len(labels), "positive_prevalence": float(positive.mean()),\n            "auroc": float(roc_auc_score(labels, scores)) if both else None,\n            "average_precision": float(average_precision_score(labels, scores)) if both else None,\n            "mcc": float(matthews_corrcoef(labels, predicted)) if both else None,\n            "precision": precision, "recall": recall, "specificity": specificity,\n            "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.,\n            "balanced_accuracy": (recall + specificity) / 2 if both else None,\n            "bce": float(-(labels * np.log(clipped) + (1 - labels) * np.log(1 - clipped)).mean()),\n            "brier": float(np.mean((scores - labels) ** 2)), "threshold": float(threshold),\n            "tp": tp, "tn": tn, "fp": fp, "fn": fn}\n', 'model.py': '"""Trainable protein-pair heads on pretrained or randomly initialized ESM-2."""\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass ResidueHead(nn.Module):\n    """Shared cross-chain attention followed by symmetric interaction pooling."""\n    def __init__(self, hidden_size, width=128, dropout=.1):\n        super().__init__()\n        self.project = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, width), nn.GELU())\n        self.attend = nn.MultiheadAttention(width, 4, dropout=dropout, batch_first=True)\n        self.combine = nn.Sequential(nn.Linear(width * 3, width), nn.GELU())\n        self.output = nn.Sequential(nn.LayerNorm(width * 2), nn.Dropout(dropout), nn.Linear(width * 2, 1))\n\n    def forward(self, states, chains):\n        features = self.project(states)\n        summaries = []\n        for own, other in [(1, 2), (2, 1)]:\n            selected = chains.eq(own)\n            attended, _ = self.attend(features, features, features, key_padding_mask=~chains.eq(other), need_weights=False)\n            matches = self.combine(torch.cat((features, attended, features * attended), -1))\n            mean = (matches * selected[..., None]).sum(1) / selected.sum(1, keepdim=True)\n            peak = matches.masked_fill(~selected[..., None], torch.finfo(matches.dtype).min).amax(1)\n            summaries.append(torch.cat((mean, peak), -1))\n        return self.output((summaries[0] + summaries[1]) * .5).squeeze(-1)\n\n\nclass PairModel(nn.Module):\n    def __init__(self, backbone, head="cls", width=128, dropout=.1, checkpointing=False):\n        super().__init__()\n        self.backbone, self.head_name = backbone, head\n        self.settings = {"head": head, "head_width": width, "dropout": dropout,\n                         "gradient_checkpointing": checkpointing}\n        size = backbone.config.hidden_size\n        if head not in {"cls", "residue"}:\n            raise ValueError("Head must be cls or residue.")\n        self.head = (nn.Sequential(nn.LayerNorm(size), nn.Dropout(dropout), nn.Linear(size, 1))\n                     if head == "cls" else ResidueHead(size, width, dropout))\n        backbone.requires_grad_(True)\n        if checkpointing:\n            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n\n    def forward(self, batch):\n        states = self.backbone.esm(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], return_dict=True).last_hidden_state\n        view_logits = (self.head(states[:, 0]).squeeze(-1) if self.head_name == "cls"\n                       else self.head(states, batch["chains"]))\n        # Log-mean-exp is a weakly supervised bag score, not a label on every crop.\n        logits = torch.stack([torch.logsumexp(view_logits[batch["owners"] == i].float(), 0)\n                              - math.log(int((batch["owners"] == i).sum())) for i in range(batch["groups"])])\n        if batch["mutation"]:\n            logits = logits.reshape(-1, 2)\n            logits = logits[:, 1] - logits[:, 0]\n        targets = batch.get("mlm_targets")\n        mlm_sum = states.new_zeros((), dtype=torch.float32)\n        if targets is not None:\n            selected = targets.ne(-100)\n            mlm_sum = F.cross_entropy(self.backbone.lm_head(states[selected]).float(), targets[selected], reduction="sum")\n        return logits, mlm_sum\n\n    def checkpoint_state(self):\n        return {name: value.detach().cpu().clone() for name, value in self.state_dict().items()}\n\n    def restore(self, state):\n        parameters = self.state_dict()\n        if set(state) != set(parameters):\n            raise ValueError("Checkpoint model tensor names differ.")\n        with torch.no_grad():\n            for name, value in state.items():\n                if value.shape != parameters[name].shape or not torch.isfinite(value).all():\n                    raise ValueError("Invalid checkpoint tensor: " + name)\n                parameters[name].copy_(value)\n', 'storage.py': '"""Verified local/Drive staging, independent of notebook-specific APIs."""\nimport json\nfrom pathlib import Path\nimport shutil\n\nfrom .base import extract_development_data, sha256\nfrom .checkpoints import CheckpointStore, atomic_json\n\n\ndef verified_copy(source, target):\n    source, target = Path(source), Path(target)\n    expected = sha256(source)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if source.resolve() == target.resolve() or target.exists() and sha256(target) == expected:\n        return target\n    temporary = target.with_suffix(target.suffix + ".partial")\n    shutil.copyfile(source, temporary)\n    if sha256(temporary) != expected or sha256(source) != expected:\n        raise OSError("Input changed or copy verification failed.")\n    temporary.replace(target)\n    return target\n\n\ndef stage_data(source, destination):\n    source, destination = Path(source), Path(destination)\n    if source.is_dir() or source.suffix == ".zip":\n        return extract_development_data(source, destination)\n    if not source.is_file() or not (source.name.endswith(".csv") or source.name.endswith(".csv.gz")):\n        raise FileNotFoundError("Supply a development CSV/CSV.gz or the existing transfer ZIP as a data-only input.")\n    return verified_copy(source, destination / source.name)\n\n\ndef restore_persistent(local, persistent):\n    """Restore the greatest available verified run; never roll local progress back."""\n    local, persistent = Path(local), Path(persistent)\n    local.mkdir(parents=True, exist_ok=True)\n    candidates = []\n    for which in ["latest", "previous"]:\n        if (persistent / (which + ".json")).exists():\n            try:\n                value = CheckpointStore(persistent).load(which, fallback=False)\n                candidates.append((value["step"], which, value["protocol"]))\n            except (RuntimeError, OSError):\n                pass\n    try:\n        existing = CheckpointStore(local).load() if (local / "latest.json").exists() else None\n    except RuntimeError:\n        existing = None\n    if not candidates:\n        if (persistent / "latest.json").exists() and existing is None:\n            raise RuntimeError("Persistent checkpoints exist but none verify; refusing to start over.")\n        return existing is not None\n    step, chosen, protocol = max(candidates, key=lambda value: value[0])\n    if existing is not None:\n        if existing["protocol"] != protocol:\n            raise ValueError("Local and persistent directories belong to different experiments.")\n        if existing["step"] >= step:\n            return True\n    for which in ["latest", "previous", "best"]:\n        selected = chosen if which == "latest" else which\n        source_pointer = persistent / (selected + ".json")\n        if not source_pointer.exists():\n            continue\n        try:\n            CheckpointStore(persistent).load(selected, fallback=False)\n        except RuntimeError:\n            continue\n        pointer = json.loads(source_pointer.read_text())\n        verified_copy(persistent / pointer["file"], local / pointer["file"])\n        atomic_json(local / (which + ".json"), pointer)\n    for name in ["run.json", "history.json", "validation.csv"]:\n        if (persistent / name).exists():\n            verified_copy(persistent / name, local / name)\n    return True\n'}
for name, source in SOURCES.items():
    (SOURCE_ROOT / "pairloom" / name).write_text(source)
for name in list(sys.modules):
    if name == "pairloom" or name.startswith("pairloom."):
        del sys.modules[name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from pairloom.storage import stage_data, restore_persistent
LOCAL_ROOT = Path("/content/pairloom-work")
DEVELOPMENT_DATA = stage_data(INPUT_DATA, LOCAL_ROOT / "inputs")
LOCAL_RUN = LOCAL_ROOT / RUN_NAME
RESUMING = restore_persistent(LOCAL_RUN, PERSISTENT_RUN)
gc.collect()
torch.cuda.empty_cache()
print("GPU:", torch.cuda.get_device_name(0))
print("Continuing verified Pairloom run" if RESUMING else "Starting NEW Pairloom run: " + INITIALIZATION)
print("Persistent checkpoints:", PERSISTENT_RUN)


In [ ]:
from pairloom.cli import main
args = ["train", "--data", str(DEVELOPMENT_DATA),
        "--output", str(LOCAL_RUN), "--persistent", str(PERSISTENT_RUN),
        "--device", "cuda", "--precision", "auto", "--head", HEAD,
        "--mlm-weight", str(MLM_WEIGHT),
        "--max-tokens", str(MAX_TOKENS), "--max-views", str(MAX_VIEWS),
        "--seed", str(SEED), "--epochs", str(EPOCHS), "--hours", str(SESSION_HOURS)]
if RESUMING:
    args.append("--resume")
else:
    args.extend(["--initialization", INITIALIZATION, "--architecture", str(ARCHITECTURE)])
    if INITIALIZATION == "pretrained" and PRETRAINED_MODEL is not None:
        args.extend(["--pretrained-model", str(PRETRAINED_MODEL)])
    if INITIALIZATION == "pretrained" and REVISION is not None:
        args.extend(["--revision", str(REVISION)])
if MAX_UPDATES is not None:
    args.extend(["--max-updates", str(MAX_UPDATES)])
main(args)
